# 08 Similarity Scale-Up

## Purpose

This notebook is my scaled-up test-set similarity pass.

The smaller manual variant notebook is still useful, but this one is for the broader questions:
- what does the full `all vs all` similarity background look like on the held-out test set?
- what happens when I take one professor-selected glycan and compare it against the entire test set?
- how big are the threshold-based similarity clouds when I pick a cutoff like `0.90` or `0.85`?
- do the nearest neighbors and the score distributions look sensible when I stop hand-picking examples?


## Setup note

Same split setup again.

- code lives in GitHub
- checkpoints, split files, metadata tables, and results live in Drive
- Colab pulls the repo first so any helper updates in `src/` show up here automatically

This notebook does one thing the earlier similarity notebook did not: it expects an accession-aware metadata table so I can start from GlyTouCan IDs and still stay tied to the test split.

In [ ]:
# ==============================================================================
# 0. SET UP THE COLAB ENVIRONMENT
# ==============================================================================
import os
import sys

from google.colab import drive

drive.mount('/content/drive')

GITHUB_OWNER = 'hb791-dev'
REPO_NAME = 'glycan-roberta'
REPO_URL = f'https://github.com/{GITHUB_OWNER}/{REPO_NAME}.git'
REPO_DIR = f'/content/{REPO_NAME}'

# Clone once in a fresh runtime. If the repo is already here, pull the latest
# version so the notebook keeps using the current helper code instead of a stale copy.
if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    print(f'Reusing existing repo at {REPO_DIR}')
    !git -C {REPO_DIR} pull --ff-only

%cd {REPO_DIR}
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)


In [ ]:
# ==============================================================================
# 1. IMPORT HELPERS AND DEFINE DRIVE PATHS
# ==============================================================================
import importlib
import json
from pathlib import Path

import pandas as pd
from IPython.display import Image, display

import src.similarity as similarity

importlib.reload(similarity)

from src.similarity import (
    build_tokenization_preview,
    load_similarity_artifacts,
    run_scaleup_similarity_analysis,
    validate_scaleup_similarity_inputs,
)

DRIVE_ROOT = Path('/content/drive/MyDrive/ProjectRoot')
CHECKPOINTS_DIR = DRIVE_ROOT / 'checkpoints'
TEST_SPLITS_DIR = DRIVE_ROOT / 'data' / 'splits'
METADATA_DIR = DRIVE_ROOT / 'data' / 'metadata'
SCALEUP_RESULTS_DIR = DRIVE_ROOT / 'results' / 'similarity_scaleup'
SCALEUP_RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print(f'Drive root: {DRIVE_ROOT}')
print(f'Checkpoints root: {CHECKPOINTS_DIR}')
print(f'Scale-up similarity results root: {SCALEUP_RESULTS_DIR}')


## Choose one model

Same idea as notebook 7: I only want to edit the model-selection cell when I switch runs.

Everything after that should stay stable so I do not accidentally change the analysis logic at the same time I change the checkpoint.

In [ ]:
# ==============================================================================
# 2. CHOOSE ONE MODEL CHECKPOINT
# ==============================================================================
MODEL_DIR = DRIVE_ROOT / 'checkpoints' / 'manual' / 'mlm15_L6_H512_A8_lr00001_ep100_setv1_train_only_v2_cont_lr5e-05_ep20' / 'best_model'

TOKENIZER_FAMILY = MODEL_DIR.parent.parent.name
EXPERIMENT_NAME = MODEL_DIR.parent.name
OUTPUT_NAME = f'{TOKENIZER_FAMILY}__{EXPERIMENT_NAME}__test_set_scaleup'
OUTPUT_DIR = SCALEUP_RESULTS_DIR / TOKENIZER_FAMILY / EXPERIMENT_NAME

print(f'Model directory: {MODEL_DIR}')
print(f'Output directory: {OUTPUT_DIR}')


## Point to the test split and the accession metadata table

This is the one slightly fussy setup part.

The split files created earlier are just plain sequence text files, which is fine for training and MLM evaluation. For this notebook, though, I also need a table that maps GlyTouCan accessions to the same compact IUPAC sequences so I can:
- find the professor-selected glycans by accession
- keep accession labels in the ranked outputs
- make the HTML reports easier to interpret later

So if this notebook errors right away, the first thing to check is whether `TEST_METADATA_PATH` points to the accession-sequence table I actually want to use.

In [ ]:
# ==============================================================================
# 3. CONFIGURE THE TEST SPLIT INPUTS
# ==============================================================================
TEST_SPLIT_PATH = TEST_SPLITS_DIR / 'test.txt'
TEST_METADATA_PATH = METADATA_DIR / 'glytoucan_sequence_table.csv'

# Change these only if the metadata file uses different header names.
ACCESSION_COL = 'accession'
SEQUENCE_COL = 'sequence'

print(f'Test split path: {TEST_SPLIT_PATH}')
print(f'Test metadata path: {TEST_METADATA_PATH}')


In [ ]:
# ==============================================================================
# 4. LOAD THE HELD-OUT TEST SET WITH ACCESSIONS
# ==============================================================================
def load_text_split(split_path: Path) -> pd.DataFrame:
    """Read one plain-text split file back into a dataframe.

    The split file itself is sequence-only. I keep a row counter so I can merge
    accession metadata onto it later without losing the original test-set order.
    """
    if not split_path.exists():
        raise FileNotFoundError(f'Test split file not found: {split_path}')

    with open(split_path, 'r', encoding='utf-8') as file:
        sequences = [line.strip() for line in file if line.strip()]

    return pd.DataFrame(
        {
            'test_row': range(1, len(sequences) + 1),
            SEQUENCE_COL: sequences,
        }
    )


def load_accession_metadata(metadata_path: Path) -> pd.DataFrame:
    """Load an accession-aware glycan table from CSV or TSV.

    I keep this flexible because I do not want the notebook structure to depend
    on one exact filename or one exact delimiter choice.
    """
    if not metadata_path.exists():
        raise FileNotFoundError(f'Accession metadata file not found: {metadata_path}')

    suffix = metadata_path.suffix.lower()
    separator = '\t' if suffix == '.tsv' else ','
    metadata_df = pd.read_csv(metadata_path, sep=separator)

    missing_columns = [column for column in [ACCESSION_COL, SEQUENCE_COL] if column not in metadata_df.columns]
    if missing_columns:
        raise ValueError(
            f'Metadata file is missing required columns: {missing_columns}. '
            f'Available columns: {metadata_df.columns.tolist()}'
        )

    metadata_df = metadata_df.copy()
    metadata_df[ACCESSION_COL] = metadata_df[ACCESSION_COL].fillna('').map(str).map(str.strip)
    metadata_df[SEQUENCE_COL] = metadata_df[SEQUENCE_COL].fillna('').map(str).map(str.strip)
    metadata_df = metadata_df.loc[metadata_df[SEQUENCE_COL] != ''].copy()
    metadata_df = metadata_df.drop_duplicates(subset=[ACCESSION_COL, SEQUENCE_COL]).reset_index(drop=True)
    return metadata_df


test_sequence_df = load_text_split(TEST_SPLIT_PATH)
accession_metadata_df = load_accession_metadata(TEST_METADATA_PATH)

# Merge the accession metadata onto the exact held-out split instead of trying to
# rebuild the split from scratch. That way the similarity corpus stays aligned with
# the same test set used everywhere else in the project.
test_glycans_df = test_sequence_df.merge(
    accession_metadata_df,
    on=SEQUENCE_COL,
    how='left',
)
test_glycans_df[ACCESSION_COL] = test_glycans_df[ACCESSION_COL].fillna('').map(str).map(str.strip)
test_glycans_df = test_glycans_df.drop_duplicates(subset=['test_row', ACCESSION_COL, SEQUENCE_COL]).reset_index(drop=True)

missing_accession_count = int(test_glycans_df[ACCESSION_COL].eq('').sum())
print(f'Test-set rows after merge: {len(test_glycans_df)}')
print(f'Rows without accession labels: {missing_accession_count}')

display(test_glycans_df.head(10))


## Choose the professor-selected glycans and the cloud thresholds

This is the main content cell for the scale-up run.

I am keeping the accession list, threshold list, and HTML display limits together here because they all affect what ends up in the final reports.

In [ ]:
# ==============================================================================
# 5. CONFIGURE THE SELECTED GLYCANS AND REPORT SETTINGS
# ==============================================================================
SELECTED_ACCESSIONS = [
    'G60230HH',
    'G74120DW',
    'G25140TA',
    'G27893KR',
]

SIMILARITY_THRESHOLDS = [0.95, 0.90, 0.85, 0.80]
ALL_VS_ALL_TOP_K = 10
HTML_NEIGHBOR_LIMIT = 50
HTML_CLOUD_LIMIT = 100
CARTOON_DEVELOPER_EMAIL = ''
CARTOON_IMAGE_FORMAT = 'svg'
LOOKUP_TIMEOUT = 60
MAX_LENGTH = None
BATCH_SIZE = 32

selected_glycans_df = test_glycans_df.loc[
    test_glycans_df[ACCESSION_COL].isin(SELECTED_ACCESSIONS)
].copy()
selected_glycans_df = selected_glycans_df.drop_duplicates(subset=[ACCESSION_COL, SEQUENCE_COL]).reset_index(drop=True)

found_accessions = selected_glycans_df[ACCESSION_COL].tolist()
missing_accessions = [accession for accession in SELECTED_ACCESSIONS if accession not in found_accessions]
if missing_accessions:
    raise ValueError(
        'Some selected glycans were not found in the accession-aware test set: '
        f'{missing_accessions}'
    )

# Reorder the dataframe to match the accession list above so the notebook displays,
# saved tables, and HTML pages all follow the same predictable order.
selected_glycans_df = (
    selected_glycans_df.set_index(ACCESSION_COL)
    .loc[SELECTED_ACCESSIONS]
    .reset_index()
)

display(selected_glycans_df[[ACCESSION_COL, SEQUENCE_COL]])


## What I expect from the outputs

The main things I want to check are:
- what the full test-set similarity background looks like when I stop hand-picking examples
- whether each selected glycan has a very tight, medium, or broad specific-vs-all distribution
- which glycans end up in the threshold clouds at `0.95`, `0.90`, `0.85`, and `0.80`
- whether the nearest-neighbor rankings look sensible enough to be worth discussing later

If the distributions are weird, that is still useful. It just means the embedding space is telling me something I need to look at more carefully.

In [ ]:
# ==============================================================================
# 6. VALIDATE INPUTS, LOAD THE MODEL, AND RUN THE ANALYSIS
# ==============================================================================
validate_scaleup_similarity_inputs(
    model_dir=MODEL_DIR,
    corpus_df=test_glycans_df[[ACCESSION_COL, SEQUENCE_COL]],
    query_df=selected_glycans_df[[ACCESSION_COL, SEQUENCE_COL]],
    accession_col=ACCESSION_COL,
    sequence_col=SEQUENCE_COL,
    output_dir=OUTPUT_DIR,
)

tokenizer, model, device = load_similarity_artifacts(str(MODEL_DIR))

# This preview is just a quick sanity check before the heavier embedding work.
selected_tokenization_preview_df = build_tokenization_preview(
    selected_glycans_df[SEQUENCE_COL].tolist(),
    tokenizer=tokenizer,
)

# The helper call does the full scale-up run: test-set embeddings, all-vs-all,
# specific-vs-all, threshold clouds, histograms, and portable HTML reports.
results = run_scaleup_similarity_analysis(
    tokenizer=tokenizer,
    model=model,
    corpus_df=test_glycans_df[[ACCESSION_COL, SEQUENCE_COL]],
    query_df=selected_glycans_df[[ACCESSION_COL, SEQUENCE_COL]],
    output_dir=OUTPUT_DIR,
    output_name=OUTPUT_NAME,
    developer_email=CARTOON_DEVELOPER_EMAIL,
    accession_col=ACCESSION_COL,
    sequence_col=SEQUENCE_COL,
    thresholds=SIMILARITY_THRESHOLDS,
    cartoon_image_format=CARTOON_IMAGE_FORMAT,
    lookup_timeout=LOOKUP_TIMEOUT,
    device=device,
    max_length=MAX_LENGTH,
    batch_size=BATCH_SIZE,
    all_vs_all_top_k=ALL_VS_ALL_TOP_K,
    html_neighbor_limit=HTML_NEIGHBOR_LIMIT,
    html_cloud_limit=HTML_CLOUD_LIMIT,
    model_dir=MODEL_DIR,
)


## All-vs-all view

This is the background landscape.

I care about it because the specific-vs-all results are easier to interpret if I also know what similarity values look like across the whole held-out set.

In [ ]:
# ==============================================================================
# 7. REVIEW THE ALL-VS-ALL OUTPUTS
# ==============================================================================
print('=== All-vs-all summary ===')
display(results['all_vs_all_artifacts']['off_diagonal_summary_df'])

print('=== All-vs-all top-neighbor preview ===')
display(results['all_vs_all_artifacts']['top_neighbors_df'].head(25))

print('=== All-vs-all histogram ===')
display(Image(filename=str(results['saved_paths']['all_vs_all_histogram_path'])))


## Specific-vs-all view

This is where the professor-selected glycans come in.

For each one, I want three related views:
- the full score distribution against the test set
- the ranked nearest-neighbor list
- the threshold-based similarity cloud

In [ ]:
# ==============================================================================
# 8. REVIEW THE SPECIFIC-VS-ALL OUTPUTS
# ==============================================================================
print('=== Selected glycan tokenization preview ===')
display(selected_tokenization_preview_df)

for accession in SELECTED_ACCESSIONS:
    print(f'=== {accession} distribution summary ===')
    display(
        results['specific_vs_all_summary_df'].loc[
            results['specific_vs_all_summary_df']['query_accession'] == accession
        ]
    )

    print(f'=== {accession} top neighbors ===')
    display(
        results['specific_vs_all_results_df'].loc[
            (results['specific_vs_all_results_df']['query_accession'] == accession)
            & (~results['specific_vs_all_results_df']['is_self_match'])
        ][['rank', 'corpus_accession', 'cosine_similarity', 'corpus_sequence']].head(15)
    )

    print(f'=== {accession} threshold cloud summary ===')
    display(
        results['threshold_summary_df'].loc[
            results['threshold_summary_df']['query_accession'] == accession
        ]
    )

    print(f'=== {accession} histogram ===')
    display(Image(filename=str(results['saved_paths']['query_histogram_paths'][accession])))


## Saved outputs

The CSVs are the structured outputs, and the HTML files are the easier-to-browse review layer.

The top-level `index.html` should be the first thing I open if I want the broad summary. The accession-specific HTML pages are where the threshold clouds live.

In [ ]:
# ==============================================================================
# 9. PRINT THE SAVED OUTPUT PATHS
# ==============================================================================
print('Saved outputs:')
for label, path in results['saved_paths'].items():
    if isinstance(path, dict):
        print(f'- {label}:')
        for child_label, child_path in path.items():
            print(f'    - {child_label}: {child_path}')
    else:
        print(f'- {label}: {path}')
